# 01 公式与算例审计（Tier A）

本 Notebook 检查论文第 5-6 页的 Eq.3-10 与附录 B 的算例。这些问题不需要训练模型，
是整份验证中最“硬”的部分：论文自己给出了输入和输出，我们只需按定义复算。

> 记号：$s$ 是原始 logits，$s^*$ 是 Eq.3 处理后的向量，$p=\mathrm{softmax}(s^*)$，
> $E_k=\sum_i \mathrm{NP}_{i,k}$ 是标签 $k$ 的能量。

$$s^*_k=\begin{cases}0,& k=\arg\max_m s_m\\ s_k,&\text{otherwise}\end{cases}\tag{Eq.3}$$

$$P_{i,k}=\frac{e^{s^*_{i,k}T}}{T},\qquad \mathrm{NP}_{i,k}=P_{i,k}-P_{i,0}\tag{Eq.4-5}$$

$$E_k=\sum_{i=1}^{N}\mathrm{NP}_{i,k},\qquad \mu=\frac{1}{K}\sum_k E_k,\qquad
\sigma^2=\frac{1}{K}\sum_k(E_k-\mu)^2\tag{Eq.6-7}$$


In [ ]:
import json
import os
import sys
from pathlib import Path

# 定位仓库根目录：容器内是 /app，本地打开时可能是 notebooks/ 或其子目录。
ROOT = Path(os.environ.get("EBBA_ROOT", Path.cwd())).resolve()
if not (ROOT / "src" / "ebba").exists():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "src" / "ebba").exists():
            ROOT = cand
            break
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("EBBA_ROOT", str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from ebba import config, nbutil
from ebba.claims import load_registry, run, run_many

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("project root:", ROOT)
print("artifacts   :", config.ARTIFACTS_DIR)
print("reports     :", config.REPORTS_DIR)
print("fast mode   :", config.FAST)


## A02：Eq.3 究竟是“置 0”还是“减最大值”？

这两个操作在 softmax 之前并不等价：减最大值是平移不变性所需的操作，置 0 不是。
论文附录 B 的示例 `[3, 8, -2] → [3, 0, -2]` 明确支持“置 0”。代码必须按字面实现，
否则后续所有能量都会与论文不同。


In [ ]:
import numpy as np
from ebba.energy import clamp_max_logits, softmax

z = np.array([[3.0, 8.0, -2.0]])
print("输入          :", z[0])
print("Eq.3 置最大项0:", clamp_max_logits(z)[0])
print("减最大值      :", (z - z.max(axis=1, keepdims=True))[0])
print("softmax(置0)  :", softmax(clamp_max_logits(z))[0])
print("softmax(减max):", softmax(z - z.max(axis=1, keepdims=True))[0])

r = nbutil.show("A02")


## A01：附录 B 的 softmax 与能量能否复算？

附录 B 给出三个样本、三个类别的 logits，并印出了 softmax 与能量。我们按论文自己的公式精确计算，
再与论文印刷值比较：

| 量 | 论文 | 精确复算 |
|---|---|---|
| $s$ 第 2 行 | `[0.12, 0.88, 0.00]` | `[0.0067, 0.9933, 0.0000]` |
| $E$（未 clamp） | `[4.82, 4.43, 4.62]` | `[4.7243, 4.7185, 4.6691]` |
| 顺序 | $E_0>E_2>E_1$ | $E_0>E_1>E_2$ |
| clamp 后 $s^*$ 第 1 行 | `[0.231, 0.66, 0.09]` | `[0.2447, 0.6652, 0.0900]` |
| clamp 后 $E$ | `[4.83, 5.54, 3.14]` | `[4.8712, 5.5436, 3.1483]` |

这里有一个容易误读的点：论文的**结论方向**（clamp 后目标类能量 $E_1^*$ 最高，因而满足 Eq.8）在复算下仍然成立；
错的是未 clamp 那张表的数值和排序。判定因此是 `CONTRADICTED`，但它否定的是“算例可复算”这一断言，
不是 EBBA 的检测机制。


In [ ]:
r = nbutil.show("A01")
print("\n逐步复算（未 clamp）：")
from ebba.claims.formulas import _np
Z = _np([[9, 1, -1], [3, 8, -2], [-1, 3, 7]])
print("softmax:\n", np.round(softmax(Z), 4))
print("E = sum_i exp(s_i):", np.round(np.exp(softmax(Z)).sum(axis=0), 4))
print("论文印刷 E       :", r.measured["energy_paper"])


## A03：三套归一化互相矛盾

论文 Eq.5 定义 $\mathrm{NP}=e^{s^*T}/T - e^{0}/T$（$T=2$ 时为 $e^{2s^*}/2-0.5$）。
官方 README 却说 CIFAR-10 上用 `NP_k = P_k - 1`；官方 Notebook 又是 `min(exp(2s)/2, 2) - 0.6`。
三者不能同时成立。下面比较同一批 logits 上的数值差异，并检查它们是否改变检测出的目标标签。


In [ ]:
import numpy as np
from ebba.energy import per_sample_normalized_energy

rng = np.random.default_rng(2024)
logits = rng.normal(0, 1.0, size=(200, 10))
logits[:, 0] += 0.8
logits[:, 1] += 0.3

variants = {v: per_sample_normalized_energy(logits, 2.0, v) for v in ("paper", "readme", "official")}
for name, arr in variants.items():
    print(f"{name:>8}: shape={arr.shape}, sum per label={np.round(arr.sum(axis=0), 3)}")

r = nbutil.show("A03")


## A04：Eq.7 的下标与分母

Eq.2/6 写 $k\in[0,K]$，即 $K+1$ 项；但类别只有 $K$ 个，Eq.7 又除以 $K$。
按字面补一个 0 项会改变 $\mu,\sigma$，从而改变 z-score 与判定阈值。实现采用 $K$ 个真实类别、除数 $K$。
下面用一个人为的能量向量展示差异大小。


In [ ]:
r = nbutil.show("A04")
print("K 项：z =", round(r.measured["zscore_K_terms_divisor_K"], 4))
print("K+1 项：z =", round(r.measured["zscore_K_plus_1_terms_divisor_K_plus_1"], 4))


## A05：TE/NTE 的恒等式与“均值为 0”

$$\mathrm{TE}=\frac{p'}{p}=e^{s'-s},\qquad \mathrm{NTE}=\mathrm{TE}-1=e^{s'-s}-1.$$

因为 softmax 每行和为 1，$\frac{1}{K}\sum_k (s'_k-s_k)=0$ 严格成立，
所以论文用基线 $e^0=1$ 有依据。下面在随机 logits 上逐项核对，并检查实现与字面公式的误差。


In [ ]:
r = nbutil.show("A05")


## A06：Eq.9/10 是否丢了温度 $T$？

Eq.4 的能量是 $e^{sT}/T$，而 Eq.9/10 的比值把 $T$ 完全消掉，等价于 $T=1$。
这不是“等价变形”，而是两处用了不同的定义。对检测标签影响多大？用 argmax 一致率衡量：

- 若一致率 ≥ 0.99，公式歧义在实践中可忽略；
- 否则说明恢复出的标签依赖温度选择。


In [ ]:
r = nbutil.show("A06")


## A17：Fig.80 的文字符号矛盾

附录 F 第一句写“值小于 0 判为后门”，紧接着又说“clean 模型小于 0、poisoned 模型大于 0”。
后者与 Eq.8（$E'_k-\mu>\lambda\sigma$ 判为后门）一致，第一句方向相反。这是一处纯文字矛盾，
代码无法“复算”，只能记录两种说法并要求以公式为准。


In [ ]:
r = nbutil.show("A17")


## A20：$\lambda=3$ 在 CIFAR-10 上正好等于统计上界

对任意 $K$ 维能量向量，单侧 $z$ 分数满足

$$\max_k \frac{E_k-\mu}{\sigma}\le\sqrt{K-1}.$$

$K=10$ 时上界恰为 $\sqrt 9=3$，且**只有当其余 9 个标签的能量完全相等时**才取到。
Eq.8 要求严格大于 $\lambda\sigma$，所以 CIFAR-10 上 `margin > 0` 在一般情况下不可能成立。
论文的 $\lambda$ 可用区间 $[1.8,5.7]$ 与 GTSRB（$K=43$，上界 6.48）相符；把 3 直接用于
CIFAR-10 时，阈值刚好压在不可达的上界上。这解释了为什么缩比实验里目标标签的 $z$ 约为
2.5-2.8，而不是 $>3$。


In [ ]:
r = nbutil.show("A20")


## 本 Notebook 的小结

| 声明 | 结论 | 要点 |
|---|---|---|
| A01 | CONTRADICTED | 附录 B 的 softmax/能量数值不可复算，排序也印错；方向性结论仍成立 |
| A02 | CONFIRMED | Eq.3 是“最大值置 0”，不是减最大值 |
| A03 | CONTRADICTED | 论文、README、官方代码是三种归一化 |
| A04 | CONTRADICTED | $K+1$ 项与除以 $K$ 不自洽 |
| A05 | CONFIRMED | TE/NTE 恒等式与均值为 0 成立 |
| A06 | CONTRADICTED | Eq.9/10 隐含 $T=1$，与 Eq.4 不一致，且影响 argmax |
| A17 | CONTRADICTED | Fig.80 首句符号写反 |
| A20 | CONTRADICTED | $\lambda=3$ 在 $K=10$ 时等于 $z$ 的上界，严格裕量不可达 |

下一步：`02_theory.ipynb` 检查 Lemma 1/2 的证明与前提。
